# BaCuSe XRD analysis

Reusable scan loading, preprocessing, reference matching, metric calculation, and plotting are implemented in `xrd_bo_optimization_tf`. Local `.xrdml` files belong in `data/raw/`; the processed metrics table is used when scans are not available.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from xrd_bo_optimization_tf.config import (
    FIGURES_DIR,
    GENERATED_XRD_METRICS_CSV,
    SYNTHESIS_CSV,
    XRD_METRICS_CSV,
)
from xrd_bo_optimization_tf.phase_matching import analyze_pattern, load_reference_cards
from xrd_bo_optimization_tf.plotting import plot_all_xrd_patterns, plot_xrd_series
from xrd_bo_optimization_tf.xrd_io import load_xrd_data

df_synth_csv = pd.read_csv(SYNTHESIS_CSV)
references = load_reference_cards()
xrd_raw = load_xrd_data(df_synth_csv['sample_id'].astype(str).tolist())
print(f'Loaded {len(xrd_raw)} XRD scans for {len(df_synth_csv)} synthesis records')

In [ ]:
xrd_results = {
    sample_id: analyze_pattern(scan['two_theta'], scan['intensity'], references)
    for sample_id, scan in xrd_raw.items()
}

metrics_path = (
    GENERATED_XRD_METRICS_CSV
    if GENERATED_XRD_METRICS_CSV.is_file()
    else XRD_METRICS_CSV
)
existing_metrics = pd.read_csv(metrics_path)
if xrd_results:
    synthesis_lookup = df_synth_csv.drop_duplicates('sample_id').set_index('sample_id')
    rows = []
    for sample_id, result in sorted(xrd_results.items()):
        if sample_id not in synthesis_lookup.index:
            continue
        synthesis = synthesis_lookup.loc[sample_id]
        rows.append({
            'sample_id': sample_id,
            'series': synthesis['series'],
            'crystallinity': result['crystallinity'],
            'primary_fwhm': result['primary_fwhm'],
            'phase_purity': result['phase_purity'],
            'n_detected_peaks': result['n_detected_peaks'],
            'n_target_matches': result['n_target_matches'],
            **{column: synthesis.get(column) for column in (
                'spin_speed_rpm', 'soft_bake_temp_c', 'hard_bake_temp_c',
                'selenization_temp_c', 'selenization_time_s',
            )},
        })
    analyzed_metrics = pd.DataFrame(rows)
    if not analyzed_metrics.empty:
        updated_ids = set(analyzed_metrics['sample_id'].astype(str))
        existing_metrics = existing_metrics.loc[
            ~existing_metrics['sample_id'].astype(str).isin(updated_ids)
        ]
        df_metrics = pd.concat([existing_metrics, analyzed_metrics], ignore_index=True)
        GENERATED_XRD_METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
        df_metrics.to_csv(GENERATED_XRD_METRICS_CSV, index=False)
        print(f'Wrote {len(analyzed_metrics)} new or updated scan metrics to {GENERATED_XRD_METRICS_CSV}')
    else:
        df_metrics = existing_metrics
        print('No XRD scans matched the synthesis table; kept the existing metrics')
else:
    df_metrics = existing_metrics
    print(f'No XRD scans found; loaded existing metrics from {metrics_path}')

df_metrics.head()

In [ ]:
series_figures = plot_xrd_series(xrd_results, df_metrics, references)
all_scans_figure = plot_all_xrd_patterns(
    xrd_results, df_metrics,
    output_path=FIGURES_DIR / 'bacuse2_xrd_all_samples.png',
)
if series_figures or all_scans_figure is not None:
    plt.show()
else:
    print('Plots will be available when XRD scans are added to data/raw/')

In [ ]:
summary_columns = [column for column in (
    'crystallinity', 'phase_purity', 'primary_fwhm', 'n_detected_peaks',
) if column in df_metrics]
df_metrics.groupby('series')[summary_columns].agg(['count', 'mean', 'min', 'max']).round(3)